In [ ]:
!pip install -q -U "torchao>=0.16.0" "peft>=0.20.0"

In [ ]:
# setup
import os
import torch
import torchao
import peft
from diffusers import StableDiffusionXLPipeline

MODEL_NAME = "stabilityai/stable-diffusion-xl-base-1.0"

LORA_PATH = (
    "/kaggle/input/models/airbwender/"
    "pytorch-lora-weights/other/default/1/"
    "pytorch_lora_weights.safetensors"
)

print("torch:", torch.__version__)
print("torchao:", torchao.__version__)
print("peft:", peft.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0))
print("LoRA exists:", os.path.exists(LORA_PATH))

In [ ]:
# load sdxl + LoRA
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
)

pipe.load_lora_weights(LORA_PATH)
pipe = pipe.to("cuda")

print("SDXL + LoRA loaded successfully!")

In [ ]:
prompts = [
    "girl, red hair, green eyes",
    "girl, blonde hair, purple eyes",
    "girl, black hair, red eyes",
    "girl, silver hair, blue eyes",
]

for prompt in prompts:

    image = pipe(
        prompt=prompt,
        height=256,
        width=256,
        num_inference_steps=25,
        guidance_scale=7.0,
    ).images[0]

    print(prompt)
    display(image)

In [ ]:
# controlled base vs LoRA experiment:
prompt = "girl, white long hair, cute"
seed = 121

pipe.unload_lora_weights()

generator = torch.Generator("cuda").manual_seed(seed)

image_base = pipe(
    prompt=prompt,
    num_inference_steps=25,
    guidance_scale=7.0,
    generator=generator,
).images[0]

display(image_base)

pipe.load_lora_weights(LORA_PATH)

generator = torch.Generator("cuda").manual_seed(seed)

image_lora = pipe(
    prompt=prompt,
    num_inference_steps=25,
    guidance_scale=7.0,
    generator=generator,
).images[0]

display(image_lora)

In [ ]:
image_lora.save("/kaggle/working/test.png")